# Stage 1-A — Pretrain (NPASS + PubChem + ZINC + ChEMBL + COCONUT, all free)

Builds a leak-proofed structure pool (natural products + public chemistry) and warm-start
pretrains the D-MPNN **body** on label-free molecular properties (no bioactivity labels needed).

Outputs: `pretrain_data/pretrain_pool.parquet`, `pretrain_data/leak_proof.csv`, `stage1a.pt`, `pretrain_data/pretrain_meta.json`.

Use with the DL notebook: `PRETRAIN_W = 'stage1a.pt'` (hook already in cell 9; load block appended to cell 31).
Gate after fine-tune: repeated-CV R2 >= 0.60, else documented negative. CPU-only by default; GPU speeds it up.

In [ ]:
!pip install --quiet rdkit pandas numpy requests gdown pyarrow

In [ ]:
# --- config (free sources; everything resumes on re-run) ---
import os, json, time, glob
import numpy as np, pandas as pd, requests

DATA_DIR = 'pretrain_data'
os.makedirs(DATA_DIR, exist_ok=True)
NPASS_PAGE = 'https://bidd.group/NPASS/downloadnpass.html'
NPASS_BASE = 'https://bidd.group/NPASS/downloadFiles/'
NPASS_CANDIDATES = {
    'structure': ['NPASS3.0_naturalproducts_structure.txt', 'NPASS3.0_naturalproducts_generalinfo.txt'],
    'activity':  ['NPASS3.0_naturalproducts_activity.txt', 'NPASS3.0_naturalproducts_activities.txt'],
}
CHEMBL_REST = 'https://www.ebi.ac.uk/chembl/api/data/activity.json'
FAMILY_TARGETS = ['CHEMBL1907611', 'CHEMBL1255126', 'CHEMBL2221344']   # MDM2, MDM4, p53-MDM4
TRAIN_CSV_CANDIDATES = ['ro5_properties_filtered.csv', 'Part_2/ro5_properties_filtered.csv',
                        'Part_15/ro5_properties_filtered.csv', '../Part_2/ro5_properties_filtered.csv',
                        '/content/ro5_properties_filtered.csv']
POOL_ROWS_MAX = 120000
LEAK_T = 0.85
RANDOM_SEED = 57
np.random.seed(RANDOM_SEED)
print('config OK - data dir:', DATA_DIR)

In [ ]:
# --- downloader (resume: skip existing) ---
def download(url, path):
    if os.path.exists(path) and os.path.getsize(path) > 0:
        print('[resume]', path, os.path.getsize(path), 'bytes'); return path
    try:
        r = requests.get(url, timeout=120, stream=True, headers={'User-Agent': 'mdm2-pretrain/1.0'})
        if r.status_code != 200:
            print('[skip]', r.status_code, url); return None
        with open(path, 'wb') as f:
            for chunk in r.iter_content(1 << 20): f.write(chunk)
        print('[ok]', path, os.path.getsize(path), 'bytes'); return path
    except Exception as e:
        print('[fail]', url, e); return None

npass_files = {}
for kind, names in NPASS_CANDIDATES.items():
    for nm in names:
        p = os.path.join(DATA_DIR, nm)
        if download(NPASS_BASE + nm, p): npass_files[kind] = p; break
if not npass_files:
    print('NPASS auto-download failed - open', NPASS_PAGE, 'and drop the structure/activity files into', DATA_DIR)
    print('(ZINC tranches: https://files.docking.org/zinc22  |  PubChem: https://ftp.ncbi.nlm.nih.gov/pubchem/)')

# --- ChEMBL family rows (IC50/Ki only, pchembl_value) ---
chembl_rows, url = [], CHEMBL_REST + '?target_chembl_id__in=' + ','.join(FAMILY_TARGETS) + '&standard_type__in=IC50,Ki&limit=1000'
while url and len(chembl_rows) < 200000:
    r = requests.get(url, timeout=120); r.raise_for_status(); j = r.json()
    for a in j.get('activities', []):
        smi = a.get('canonical_smiles'); pv = a.get('pchembl_value')
        if smi and pv: chembl_rows.append({'smiles': smi, 'pchembl': float(pv),
                                           'target': a.get('target_chembl_id'), 'source': 'chembl_family'})
    url = j.get('page_meta', {}).get('next'); print('chembl rows:', len(chembl_rows), flush=True)
chembl_df = pd.DataFrame(chembl_rows).drop_duplicates('smiles') if chembl_rows else pd.DataFrame(columns=['smiles','pchembl','target','source'])
print('ChEMBL family pool:', len(chembl_df))
chembl_df.to_csv(os.path.join(DATA_DIR, 'chembl_family.csv'), index=False)
print('local files present:', [os.path.basename(p) for p in glob.glob(os.path.join(DATA_DIR, '*')) if 'chembl_family.csv' not in p][:20])

In [ ]:
# --- parse NPASS structures (detect smiles + inchikey cols) ---
from rdkit import Chem
from rdkit.Chem import inchi
from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')

def read_any(path):
    try: return pd.read_csv(path, sep=None, engine='python', dtype=str, on_bad_lines='skip')
    except Exception: return pd.read_csv(path, sep=r'\t', dtype=str, on_bad_lines='skip')

npass = pd.DataFrame()
if npass_files.get('structure'):
    raw = read_any(npass_files['structure'])
    cols = {c.lower(): c for c in raw.columns}
    smi_col = next((cols[k] for k in cols if 'smile' in k), None)
    key_col = next((cols[k] for k in cols if 'inchikey' in k or 'inchi_key' in k), None)
    if smi_col:
        npass = raw[[smi_col]].rename(columns={smi_col: 'smiles'})
        if key_col: npass['inchikey'] = raw[key_col]
        npass['source'] = 'npass'
        print('NPASS structures:', len(npass), '| cols seen:', list(raw.columns)[:12])
    else:
        print('NPASS structure file: no SMILES column found - columns:', list(raw.columns)[:20])
else:
    print('no NPASS structure file yet (manual download needed - see cell above)')

In [ ]:
# --- pool assembly: NPASS + ChEMBL + local *.smi/*.csv samples (ZINC/PubChem drops) ---
pool = []
if len(npass): pool.append(npass)
if len(chembl_df): pool.append(chembl_df[['smiles']].assign(source='chembl_family'))
for p in glob.glob(os.path.join(DATA_DIR, '*.smi')) + glob.glob(os.path.join(DATA_DIR, '*.smi.gz')) + glob.glob(os.path.join(DATA_DIR, '*sample*.csv')):
    try:
        if p.endswith(('.smi', '.smi.gz')):
            dfx = pd.read_csv(p, sep=r'\s+', header=None, names=['smiles'], dtype=str, nrows=POOL_ROWS_MAX)
        else:
            dfx = read_any(p); c = {cc.lower(): cc for cc in dfx.columns}; sc = next((c[k] for k in c if 'smile' in k), dfx.columns[0]); dfx = dfx[[sc]].rename(columns={sc: 'smiles'})
        dfx['source'] = os.path.basename(p)
        pool.append(dfx[['smiles', 'source']])
        print('added local sample:', p, len(dfx))
    except Exception as e: print('[skip local]', p, e)
pool = pd.concat(pool, ignore_index=True) if pool else pd.DataFrame(columns=['smiles', 'source'])
pool = pool.dropna(subset=['smiles']).drop_duplicates('smiles')
print('pool raw:', len(pool), pool['source'].value_counts().to_dict() if len(pool) else {})

In [ ]:
# --- InChIKey dedupe + validity (chunked, CPU) ---
keys, good = [], []
for i, s in enumerate(pool['smiles'].tolist()):
    m = Chem.MolFromSmiles(s)
    if m is None: continue
    try: k = inchi.MolToInchiKey(m)
    except Exception: continue
    keys.append(k); good.append(i)
    if (i + 1) % 25000 == 0: print('parsed', i + 1, '/', len(pool), flush=True)
pool = pool.iloc[good].reset_index(drop=True); pool['inchikey'] = keys
pool = pool.drop_duplicates('inchikey').reset_index(drop=True)
print('pool after dedupe:', len(pool))

In [ ]:
# --- L1/L6 leak filter vs the 645 (Tanimoto >= 0.85 | Murcko | InChIKey) - with proof ---
from rdkit.Chem.Scaffolds import MurckoScaffold
from rdkit.Chem import rdFingerprintGenerator

df = None
for p in TRAIN_CSV_CANDIDATES:
    if os.path.exists(p): df = pd.read_csv(p); print('train loaded:', p); break
assert df is not None, 'train 645 CSV not found - adjust TRAIN_CSV_CANDIDATES'
smi_col = 'canonical_smiles' if 'canonical_smiles' in df.columns else [c for c in df.columns if 'smile' in c.lower()][0]
train_smi = df[smi_col].astype(str).tolist()
mg = rdFingerprintGenerator.GetMorganGenerator(radius=3, fpSize=512)
Tfp = np.stack([np.array(mg.GetFingerprint(Chem.MolFromSmiles(s)), dtype=np.uint8) for s in train_smi]).astype(np.float32)
Tkey = set(); Tmurcko = set()
for s in train_smi:
    m = Chem.MolFromSmiles(s)
    if m:
        try: Tkey.add(inchi.MolToInchiKey(m))
        except Exception: pass
        Tmurcko.add(MurckoScaffold.MurckoScaffoldSmiles(mol=m))

pkeys = pool['inchikey'].tolist(); psmi = pool['smiles'].tolist()
dropped = np.zeros(len(pool), dtype=bool); maxT = np.zeros(len(pool), dtype=np.float32); nnidx = np.full(len(pool), -1, np.int32)
CH = 5000
for s in range(0, len(pool), CH):
    e = min(len(pool), s + CH)
    Q = np.stack([np.array(mg.GetFingerprint(Chem.MolFromSmiles(x)), dtype=np.uint8) if Chem.MolFromSmiles(x) else np.zeros(512, np.uint8) for x in psmi[s:e]]).astype(np.float32)
    S = Q @ Tfp.T; qc = Q.sum(1)
    D = qc[:, None] + Tfp.sum(1)[None, :] - S
    T = S / np.maximum(D, 1e-9); j = T.argmax(1)
    maxT[s:e] = T[np.arange(e - s), j]; nnidx[s:e] = j
    print('leak-scan', e, '/', len(pool), flush=True)

for i in range(len(pool)):
    if pkeys[i] in Tkey: dropped[i] = True
    elif maxT[i] >= LEAK_T: dropped[i] = True
    else:
        m = Chem.MolFromSmiles(psmi[i])
        if m and MurckoScaffold.MurckoScaffoldSmiles(mol=m) in Tmurcko: dropped[i] = True

proof = pd.DataFrame({'pool_inchikey': pkeys, 'nearest_train_Tanimoto_r3_512': maxT, 'kept': ~dropped})
proof.sort_values('nearest_train_Tanimoto_r3_512', ascending=False).to_csv(os.path.join(DATA_DIR, 'leak_proof.csv'), index=False)
print('pool:', len(pool), '| leaked/dropped:', int(dropped.sum()), '| kept:', int((~dropped).sum()))
pool = pool[~dropped].reset_index(drop=True)

In [ ]:
# --- cap + save pool ---
if len(pool) > POOL_ROWS_MAX:
    pool = pool.sample(POOL_ROWS_MAX, random_state=RANDOM_SEED).reset_index(drop=True)
pool.to_parquet(os.path.join(DATA_DIR, 'pretrain_pool.parquet'), index=False)
meta = {'rows': int(len(pool)), 'sources': pool['source'].value_counts().to_dict() if len(pool) else {},
        'leak_t': LEAK_T, 'rows_max': POOL_ROWS_MAX, 'time': time.strftime('%Y-%m-%d %H:%M')}
json.dump(meta, open(os.path.join(DATA_DIR, 'pool_meta.json'), 'w'), indent=2)
print('saved pretrain_pool.parquet:', meta)

## Warm-start pretrain (label-free properties)

Extracts the EXACT featurizer + DirectedMPNN classes from the patched DL notebook, then trains on
RDKit-computed properties (no bioactivity labels). Fine-tuning stays in the DL notebook.

In [ ]:
# --- extract featurizer + model classes from the DL notebook (guarantees compatibility) ---
import json as _json
NB_PATHS = ['testing/fix_of_chemphore/colabtestregressiom_fast.ipynb', 'colabtestregressiom_fast.ipynb',
            '/content/colabtestregressiom_fast.ipynb', '../testing/fix_of_chemphore/colabtestregressiom_fast.ipynb']
nb = None
for p in NB_PATHS:
    if os.path.exists(p): nb = _json.load(open(p)); print('DL nb loaded:', p); break
assert nb is not None, 'DL notebook not found - adjust NB_PATHS'
wanted = ['def one_hot', 'def atom_features', 'def bond_features', 'def mol_to_graph', 'class DirectedMPNN']
srcs = ['\n'.join(c['source']) for c in nb['cells'] if c['cell_type'] == 'code' and any(w in ''.join(c['source']) for w in wanted)]
exec(compile('\n\n'.join(srcs), 'dl_arch', 'exec'), globals())
print('arch extracted: DirectedMPNN + featurizer ready')
g = mol_to_graph('CC(=O)Oc1ccccc1C(=O)O', 0.0)
assert g.x.shape[1] == 124, f'unexpected feature dim {g.x.shape[1]}'
print('feature dim check OK (124)')

In [ ]:
# --- build pretrain tensors: label-free properties ---
from rdkit.Chem import Descriptors
import torch
from torch_geometric.data import DataLoader as PyGLoader

pool = pd.read_parquet(os.path.join(DATA_DIR, 'pretrain_pool.parquet'))
MAX_TRAIN_ROWS = 60000   # CPU-safe v1; raise on GPU
pool = pool.sample(min(len(pool), MAX_TRAIN_ROWS), random_state=RANDOM_SEED).reset_index(drop=True)
graphs, props = [], []
for s in pool['smiles']:
    m = Chem.MolFromSmiles(s)
    if m is None: continue
    g = mol_to_graph(s, 0.0)
    if g is None: continue
    graphs.append(g)
    props.append([Descriptors.MolWt(m), Descriptors.MolLogP(m), Descriptors.TPSA(m),
                  Descriptors.NumHDonors(m), Descriptors.NumHAcceptors(m), Descriptors.RingCount(m)])
props = np.asarray(props, dtype=np.float32)
from sklearn.preprocessing import StandardScaler
pscaler = StandardScaler().fit(props)
props_s = pscaler.transform(props)
for g, pv in zip(graphs, props_s): g.y = torch.tensor(pv, dtype=torch.float).view(1, -1)
loader = PyGLoader(graphs, batch_size=64, shuffle=True)
print('pretrain graphs:', len(graphs), '| property targets:', props.shape[1])

In [ ]:
# --- warm-start pretrain loop (multi-task property regression; 4h wall-clock cap) ---
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
try:
    model = DirectedMPNN(num_node_features=124, num_edge_features=6, hidden_dim=64, depth=2, dropout=0.2, out_dim=6)
    print('multi-task head OK (out_dim=6)')
except TypeError:
    model = DirectedMPNN(num_node_features=124, num_edge_features=6, hidden_dim=64, depth=2, dropout=0.2, out_dim=1)
    print('single-task fallback head (out_dim=1)')
model = model.to(device)
opt = torch.optim.Adam(model.parameters(), lr=3e-4, weight_decay=1e-5)
crit = torch.nn.SmoothL1Loss()
T0 = time.time(); WALL = 4 * 3600; steps = 0; history = []
for epoch in range(2):
    model.train()
    for batch in loader:
        batch = batch.to(device)
        y = batch.y if batch.y.dim() == 2 else batch.y.view(-1, 1)
        out = model(batch.x, batch.edge_index, batch.edge_attr, batch.batch)
        if getattr(out, 'dim', lambda: 0)() == 2 and y.shape[1] > 1: y = y[:, :1]
        if out.shape[-1] != y.shape[-1]: y = y[:, :1]
        loss = crit(out, y)
        opt.zero_grad(); loss.backward(); opt.step(); steps += 1
        if steps % 200 == 0:
            history.append({'step': steps, 'loss': float(loss.detach().cpu()), 'min': (time.time() - T0) / 60})
            print(f'ep{epoch} step {steps} loss {loss.item():.4f} ({(time.time()-T0)/60:.1f} min)', flush=True)
        if time.time() - T0 > WALL:
            print('wall-clock cap reached - stopping'); break
    if time.time() - T0 > WALL: break

torch.save({'model': model.state_dict(), 'out_dim': int(out.shape[-1]), 'featurizer': '124-dim (DL nb patched)',
            'rows': len(graphs), 'steps': steps}, 'stage1a.pt')
json.dump({'rows': len(graphs), 'steps': steps, 'history': history[-30:], 'wall_min': (time.time() - T0) / 60},
          open(os.path.join(DATA_DIR, 'pretrain_meta.json'), 'w'), indent=2)
print('saved stage1a.pt - set PRETRAIN_W = "stage1a.pt" in the DL notebook (cell 9) for warm-start fine-tuning.')

In [ ]:
# --- validation + usage ---
print('SUMMARY')
print(' pool rows:', len(pool))
print(' leak proof: pretrain_data/leak_proof.csv (nearest Tanimoto to the 645 per pool compound)')
print(' pretrain meta:', json.load(open(os.path.join(DATA_DIR, 'pretrain_meta.json'))))
print(' weights: stage1a.pt')
print(' next: DL notebook -> PRETRAIN_W="stage1a.pt", run fine-tune, gate = repeated-CV R2 >= 0.60')